# 00 · Setup Environment dan Verifikasi Dataset

**Proyek:** Prediksi Risiko Diabetes Menggunakan Multilayer Perceptron (MLP)

**Mata kuliah:** Pembelajaran Mendalam · Tugas 1 & UTS

**Kelompok:** Mythos

## Deskripsi
Notebook ini menyiapkan lingkungan kerja Google Colab yang seragam untuk seluruh anggota kelompok sebelum masuk ke EDA dan pemodelan.

## Tujuan
1. Memastikan repository bisa di-clone dan modul `src/` bisa di-import.
2. Mencatat versi Python dan library (bahan bab **A.7 Teknologi**).
3. Memastikan seed bekerja sehingga eksperimen bisa diulang.
4. Mengunduh dataset dan memverifikasi bahwa file, ukuran, dan kolomnya sesuai.
5. Mencatat checksum (MD5) dataset supaya semua anggota terbukti memakai file yang identik.

## Kriteria selesai
Kelima anggota berhasil menjalankan notebook ini dari atas sampai bawah tanpa error, dan nilai MD5 yang muncul **sama** di semua akun.

In [12]:
!pwd
!ls -la

/content
total 20
drwxr-xr-x  1 root root 4096 Oct 30 16:29 .
drwxr-xr-x  1 root root 4096 Oct 30 16:27 ..
drwxr-xr-x  4 root root 4096 Oct 26 22:18 .config
drwxr-xr-x 11 root root 4096 Oct 30 16:30 diabetes-risk-mlp
drwxr-xr-x  1 root root 4096 Oct 26 22:18 sample_data


In [15]:
%cd /content/diabetes-risk-mlp

/content/diabetes-risk-mlp


In [16]:
!git config user.name "JohannesHartanto"
!git config user.email "johannes.hartanto@students.mikroskil.ac.id"

!git add .
!git commit -m "Tes Commit"

[main d8fbbcb] Tes Commit
 1 file changed, 31 insertions(+)
 create mode 100644 results/environment.json


## 1. Bootstrap Proyek

**Tujuan:** menyiapkan path agar kode di folder `src/` bisa dipakai dari notebook.

**Alasan:** definisi kolom, urutan fitur, dan seed disimpan di satu tempat (`src/config.py`). Kalau setiap notebook menulis ulang definisinya sendiri, risiko angka tidak konsisten antarartefak menjadi besar.

> Sebelum menjalankan, ganti `REPO_URL` dengan URL repository kelompok. Repository harus **publik**.

In [1]:
# ============================================================
# Bootstrap: menyiapkan path proyek (Colab maupun lokal)
# ============================================================
import os
import subprocess
import sys
from pathlib import Path

# GANTI sekali saja dengan URL repository GitHub kelompok (harus PUBLIK)
REPO_URL = "https://github.com/felix-malvin/mythos-diabetes-risk-mlp.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content/diabetes-risk-mlp")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)
else:
    # Lokal: naik folder sampai menemukan folder src/
    ROOT = Path.cwd().resolve()
    while not (ROOT / "src").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Berjalan di Colab :", IN_COLAB)
print("Root proyek       :", ROOT)

Berjalan di Colab : True
Root proyek       : /content/diabetes-risk-mlp


## 2. Instalasi Library Tambahan

**Tujuan:** memasang `kagglehub` untuk mengunduh dataset langsung dari Kaggle.

**Alasan:** library lain (NumPy, pandas, scikit-learn, TensorFlow, Matplotlib, seaborn) sudah tersedia di Colab. Tidak perlu memasang ulang supaya versinya tetap mengikuti Colab dan tercatat apa adanya.

In [2]:
%pip install -q kagglehub

## 3. Import Library dan Konfigurasi

In [3]:
import json
import platform

import numpy as np
import pandas as pd

from src.config import (CSV_NAME, EXPECTED_SHAPE, FEATURES, KAGGLE_DATASET,
                        RESULTS_DIR, SEED, TARGET)
from src.data import file_md5, get_raw_path, load_raw
from src.utils import get_versions, set_seed

pd.set_option("display.max_columns", 50)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Seed proyek:", SEED)

Seed proyek: 42


## 4. Pemeriksaan Runtime dan Hardware

**Tujuan:** mencatat spesifikasi runtime (CPU/GPU) untuk bab **A.7.c Lingkungan Pengembangan**.

**Catatan:** MLP pada data tabular berukuran ratusan ribu baris tetap bisa dilatih di CPU. GPU mempercepat, tetapi tidak wajib. Yang penting, runtime yang dipakai untuk **pelatihan final** dicatat di laporan.

In [4]:
runtime = {"platform": platform.platform(), "processor": platform.processor() or "tidak terbaca"}

try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    runtime["gpu_tersedia"] = len(gpus) > 0
    runtime["gpu_devices"] = [g.name for g in gpus]
except ImportError:
    runtime["gpu_tersedia"] = None
    runtime["gpu_devices"] = "TensorFlow tidak terpasang"

# Nama GPU lengkap (misalnya Tesla T4) hanya bisa dibaca lewat nvidia-smi
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, check=True)
    runtime["gpu_nvidia_smi"] = out.stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    runtime["gpu_nvidia_smi"] = "tidak ada GPU NVIDIA"

for k, v in runtime.items():
    print(f"{k:16s}: {v}")

platform        : Linux-6.6.159+-x86_64-with-glibc2.39
processor       : x86_64
gpu_tersedia    : False
gpu_devices     : []
gpu_nvidia_smi  : tidak ada GPU NVIDIA


## 5. Pencatatan Versi Library

**Tujuan:** menghasilkan daftar versi untuk laporan (bab A.7.b) dan untuk mem-*pin* `requirements.txt`.

**Alasan:** Colab memperbarui versi library secara berkala. Tanpa catatan versi, hasil yang sama belum tentu bisa direproduksi beberapa minggu kemudian.

In [5]:
versions = get_versions()
pd.DataFrame(versions.items(), columns=["Komponen", "Versi"])

,Komponen,Versi
0,python,3.13.18
1,numpy,2.1.3
2,pandas,2.2.3
3,sklearn,1.6.1
4,scipy,1.16.3
5,matplotlib,3.10.0
6,seaborn,0.13.2
7,tensorflow,2.21.0
8,keras,3.13.2
9,kagglehub,1.0.3


## 6. Uji Reproducibility Seed

**Tujuan:** membuktikan bahwa `set_seed()` menghasilkan angka acak yang sama setiap kali dipanggil.

**Cara membaca:** kedua baris angka di bawah harus identik dan hasil akhirnya `True`. Kalau `False`, ada sumber acak yang belum terkunci.

In [6]:
def sample_random_numbers():
    set_seed(SEED)
    return np.random.rand(5)

first, second = sample_random_numbers(), sample_random_numbers()
print("Percobaan 1:", np.round(first, 6))
print("Percobaan 2:", np.round(second, 6))
print("Identik    :", np.array_equal(first, second))

Percobaan 1: [0.37454  0.950714 0.731994 0.598658 0.156019]
Percobaan 2: [0.37454  0.950714 0.731994 0.598658 0.156019]
Identik    : True


## 7. Unduh dan Verifikasi Dataset

**Tujuan:** memastikan file yang dipakai benar-benar `diabetes_binary_health_indicators_BRFSS2015.csv` dengan ukuran dan kolom sesuai dokumentasi.

**Cara kerja `load_raw()`:**
1. Mencari CSV di `data/raw/`. Jika belum ada, mengunduh lewat `kagglehub` lalu menyalinnya ke `data/raw/`.
2. Memeriksa nama kolom, ukuran data, dan rentang nilai setiap fitur sesuai codebook.
3. Mengubah tipe data dari float (format asli CSV) ke integer, setelah dipastikan tidak ada nilai pecahan.

**Jika unduhan gagal:** isi `KAGGLE_USERNAME` dan `KAGGLE_KEY` di menu *Secrets* Colab (ikon kunci di sidebar), atau unduh CSV manual dari Kaggle lalu unggah ke `/content/diabetes-risk-mlp/data/raw/`.

In [7]:
df = load_raw(verify_md5=False)
raw_path = get_raw_path()
data_md5 = file_md5(raw_path)

print("File        :", raw_path.name)
print("Ukuran      :", df.shape, "| diharapkan:", EXPECTED_SHAPE, "| sesuai:", df.shape == EXPECTED_SHAPE)
print("Target      :", TARGET)
print("Jumlah fitur:", len(FEATURES))
print("MD5         :", data_md5)
df.head()

Using Colab cache for faster access to the 'diabetes-health-indicators-dataset' dataset.
File        : diabetes_binary_health_indicators_BRFSS2015.csv
Ukuran      : (253680, 22) | diharapkan: (253680, 22) | sesuai: True
Target      : Diabetes_binary
Jumlah fitur: 21
MD5         : 8f90e44047182c5f32d10f0c4dd4890f


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1,1,1,40,1,0,0,0,0,1,0,1,0,5,18,15,1,0,9,4,3
1,0,0,0,0,25,1,0,0,1,0,0,0,0,1,3,0,0,0,0,7,6,1
2,0,1,1,1,28,0,0,0,0,1,0,0,1,1,5,30,30,1,0,9,4,8
3,0,1,0,1,27,0,0,0,1,1,1,0,1,0,2,0,0,0,0,11,3,6
4,0,1,1,1,24,0,0,0,1,1,1,0,1,0,2,3,0,0,0,11,5,4


In [8]:
# Tipe data setelah konversi. Semua kolom seharusnya int64.
df.dtypes.value_counts()

,count
int64,22


## 8. Simpan Catatan Environment

**Tujuan:** menyimpan seluruh informasi di atas ke `results/environment.json`. File ini menjadi sumber tunggal untuk bab Teknologi di laporan Tugas 1 dan UTS.

In [9]:
environment = {
    "runtime": runtime,
    "versions": versions,
    "seed": SEED,
    "dataset": {
        "kaggle": KAGGLE_DATASET,
        "file": CSV_NAME,
        "shape": list(df.shape),
        "md5": data_md5,
    },
}
env_path = RESULTS_DIR / "environment.json"
env_path.write_text(json.dumps(environment, indent=2, ensure_ascii=False))
print("Tersimpan:", env_path)

Tersimpan: /content/diabetes-risk-mlp/results/environment.json


In [10]:
# Mengemas results/ menjadi zip supaya grafik bisa diunggah ke branch GitHub masing-masing
import shutil
from src.config import RESULTS_DIR

archive = shutil.make_archive(str(ROOT / "results_export"), "zip", RESULTS_DIR)
print("Arsip dibuat:", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)

Arsip dibuat: /content/diabetes-risk-mlp/results_export.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Interpretasi

- **Seed:** jika hasil uji di bagian 6 bernilai `True`, sumber acak NumPy dan Python sudah terkunci. TensorFlow ikut dikunci oleh `set_seed()` pada notebook pelatihan.
- **Dataset:** ukuran yang sesuai (`253.680 × 22`) dan tidak adanya peringatan rentang nilai berarti file yang dipakai sama dengan yang didokumentasikan di Kaggle.
- **MD5:** angka ini adalah "sidik jari" file. Jika MD5 di akun semua anggota sama, maka semua eksperimen berangkat dari data yang identik.

## 10. Kesimpulan dan Tindak Lanjut

Environment siap dipakai untuk tahap EDA (`01_eda.ipynb`).

**Yang perlu dilakukan setelah notebook ini selesai:**
1. Setiap anggota membandingkan nilai MD5 di grup. Harus sama.
2. Felix menyalin nilai MD5 ke `DATA_MD5` di `src/config.py` lewat satu PR, supaya notebook berikutnya menolak file yang berbeda.
3. Felix mem-*pin* versi di `requirements.txt` berdasarkan `results/environment.json`.
4. Setiap anggota menyimpan notebook ini ke **branch masing-masing** sebagai commit pertama (*File → Save a copy in GitHub*).

**Kirimkan ke mentor:** isi `results/environment.json` (atau screenshot tabel versi dan output bagian 7).

## Referensi
- Teboul, A. *Diabetes Health Indicators Dataset*. Kaggle. https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset
- Centers for Disease Control and Prevention. *Behavioral Risk Factor Surveillance System 2015 Survey Data*. https://www.cdc.gov/brfss/annual_data/annual_2015.html
- TensorFlow. *tf.keras.utils.set_random_seed*. https://www.tensorflow.org/api_docs/python/tf/keras/utils/set_random_seed